# 04 NON-DP CTGAN Synthesis and Fairness Auditing

In this notebook, we'll demonstrate how to synthesize a NON-DP audit table using `CTGANSynthesizer`, and then use it to audit both a pre-trained `Logistic Regression` and `FT-Transformer` model for fairness.

> **Note**: This notebook expects the models to have already been trained by `01_train_models_step_by_step.ipynb`. If they are not found, it will raise an error asking you to run it first.

In [1]:
import sys
import os
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path(".").resolve().parent / "src"))

import numpy as np
import pandas as pd
import joblib

from fairness_auditbench.datasets.folktables_acs import ACSPublicCoverageDataset
from fairness_auditbench.synthesizers.registry import get_synthesizer
import fairness_auditbench.synthesizers.ctgan  # Ensure it registers
from fairness_auditbench.metrics.fairness import compute_fairness_metrics
from fairness_auditbench.runners.audit import evaluate_model

## 1. Data Preparation
We use the full dataset corresponding to the trained model (default CA, 2018).

In [2]:
dataset = ACSPublicCoverageDataset(
    states=["CA"], 
    year=2018, 
    fast_dev_run=False
)

train_df, val_df, test_df, spec = dataset.get_splits(seed=0)
print(f"Test set size: {len(test_df)}")

Test set size: 27712


## 2. Check for Pre-trained Models
Before synthesizing data, we verify that the models trained in Notebook 01 actually exist.

In [3]:
# This matches the output directory of train_model.py for the full dataset
model_dir_logreg = Path("../results/models/acs_public_coverage/logreg/seed=0")
model_dir_ft = Path("../results/models/acs_public_coverage/ft_transformer/seed=0")

for m_dir in [model_dir_logreg, model_dir_ft]:
    if not m_dir.exists():
        raise FileNotFoundError(
            f"Model not found at {m_dir}. "
            "Please run Notebook 01_train_models_step_by_step.ipynb first to train the model!"
        )
print("Found pre-trained Logistic Regression and FT-Transformer models.")

Found pre-trained Logistic Regression and FT-Transformer models.


## 3. Synthesize Audit Table with NON-DP CTGAN
Using the NON-DP synthesizer to create a synthetic version of the  split. We run it via .

In [8]:
epsilon = None
delta = None
seed = 0    

synth = get_synthesizer("ctgan", epochs=20, batch_size=500)
synth.fit(test_df, spec=spec, epsilon=epsilon, delta=delta, seed=seed)
synth_df = synth.sample(n=len(test_df), seed=seed)

print("Synthetic data generated successfully.")

/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


Synthetic data generated successfully.


## 4. Audit Fairness (Logistic Regression)
Load our trained Logistic Regression model and compute predictions on both real and synthetic data.

In [5]:
# Real Predictions
real_preds_df_logreg = evaluate_model(test_df, "logreg", model_dir_logreg, spec)
real_metrics_logreg = compute_fairness_metrics(
    real_preds_df_logreg, spec.label_col, "_y_pred", spec.sensitive_col
)

# Synthetic Predictions
synth_preds_df_logreg = evaluate_model(synth_df, "logreg", model_dir_logreg, spec)
synth_metrics_logreg = compute_fairness_metrics(
    synth_preds_df_logreg, spec.label_col, "_y_pred", spec.sensitive_col
)

## 5. Audit Fairness (FT-Transformer)
Load our trained FT-Transformer model and compute predictions.

In [6]:
# Real Predictions
real_preds_df_ft = evaluate_model(test_df, "ft_transformer", model_dir_ft, spec)
real_metrics_ft = compute_fairness_metrics(
    real_preds_df_ft, spec.label_col, "_y_pred", spec.sensitive_col
)

# Synthetic Predictions
synth_preds_df_ft = evaluate_model(synth_df, "ft_transformer", model_dir_ft, spec)
synth_metrics_ft = compute_fairness_metrics(
    synth_preds_df_ft, spec.label_col, "_y_pred", spec.sensitive_col
)

/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/tmp/python-venv/fairness-auditbench_venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


## 6. View Audit Error
We compute the difference between the fairness metric measured on the real data vs the synthetic data.

In [7]:
def print_audit_error(model_name, real_m, synth_m):
    audit_error = {
        k: abs(real_m.get(k, 0.0) - synth_m.get(k, 0.0))
        for k in ["demographic_parity_score", "equal_opportunity_score", "equalized_odds_score"]
    }
    
    print(f"\n{'='*15} {model_name} {'='*15}")
    print("--- REAL METRICS ---")
    for k, v in real_m.items():
        if not isinstance(v, dict):
            print(f"{k}: {v:.4f}")
            
    print("\n--- SYNTH METRICS ---")
    for k, v in synth_m.items():
        if not isinstance(v, dict):
            print(f"{k}: {v:.4f}")
            
    print("\n--- AUDIT ERRORS ---")
    for k, v in audit_error.items():
        print(f"{k}: {v:.4f}")

print_audit_error("Logistic Regression", real_metrics_logreg, synth_metrics_logreg)
print_audit_error("FT-Transformer", real_metrics_ft, synth_metrics_ft)



=============== Logistic Regression ===============
--- REAL METRICS ---
demographic_parity_score: 0.3414
equal_opportunity_score: 0.1359
equalized_odds_score: 0.4194

--- SYNTH METRICS ---
demographic_parity_score: 0.1333
equal_opportunity_score: 0.2083
equalized_odds_score: 0.2083

--- AUDIT ERRORS ---
demographic_parity_score: 0.2080
equal_opportunity_score: 0.0725
equalized_odds_score: 0.2110

=============== FT-Transformer ===============
--- REAL METRICS ---
demographic_parity_score: 0.3313
equal_opportunity_score: 0.3057
equalized_odds_score: 0.4086

--- SYNTH METRICS ---
demographic_parity_score: 0.1633
equal_opportunity_score: 0.2125
equalized_odds_score: 0.2125

--- AUDIT ERRORS ---
demographic_parity_score: 0.1680
equal_opportunity_score: 0.0932
equalized_odds_score: 0.1961
